DEPLOYMENT DI RESNET CON FLASK

Deployment di ResNet con Flask significa prendere un modello di Computer Vision già addestrato e trasformarlo in un servizio utilizzabile da altre applicazioni.

L'obbiettivo finale di AI non è la prove sul proprio pc, ma utilizzerlo.
Vediamo come trasformare un file di pesi in un servizio web capace di vedere, rispondere a chiunque nel mondo.

- Persistenza del modello: salvataggio e gestione degli artifact
- Interfaccia REST: creazione di un server Flask per l'inferenza di immagini via HTTP
- Ottimizzazione produttiva: gestione della concorrenza e server WSGI per il carico reale senza andare in crash.

modello ResNet -> Flask API -> HTTP request con immagine -> preprocessing -> model.predict() -> risultato -> json

ResNet fa la classifizione, Flask non fa AI, è semplicimente il livello applicativo che permette ad un altro programma, di chiamare il modello.

Durante lo sviluppo hai qualcosa tipo:
model=...
model.fit(...)
model.save("modello.kera")

E qui completi il training
Il deployment parte dopo

TRAINING
    immagini -> model.fit()
        -
    modelello.keras

DEPLOYMENT
    modello.kera
        -
    applicazione Flask
        -
    predizioni

Flask deve fare inferenza non addestramentoo.

Cosè Flask?
Flask è un framework web di Python
Flask ricveve l'immagine, chiama ResNet, restituisce il risultato.

Quindi puoi richiamare il modello da Python, JavaScript, un'app monile, un ERP o qualsiasi altro programma capace di effetturare richieste HTTP

Dopo avere caricato il modello e salvato su Flask, per utilizzarlo lo carichi una sola volta in RAM e poi fai .predict tutte le volte che vuoi

Come congeliamo l'intelligenze che abbiamo addestrato?

Salvataggio e Persistenza con .keras
Dal training alla produzione: esportare l'intelligenza
Una volta terminato il fine-tuning di una ResNet, il modello risiede in RAM del nostro ambiente di sviluppo, volatile come un ricordo. Per renderla utilizzabile in un'applicazione esterna, dobbiamo serializzarlo in un formato che ne conservi l'architettura logica, pesi e configurazione dell'ottimizzatore. E' come scrivere la ricetta completa di un piatto in modo che chiunque possa replicarlo esatttamente nello stesso modo.

Il nuovo formato nativo '.keras' di Keras3 è lo standard raccomandato, superando i limiti del vecchio file '.h5' grazie a una struttura basata su archivi ZIP che garantisce maggiore portabilità e sicurezza.
E' autocontenuto (non dovete preoccuparvi di prendere pezzi per strada) inoltre un modello salvato con TensorFlow, può essere ricaricato con altri motori, è la massima espressioe della portabilità.

Ma un modello, da solo, non basta, ha bisogno di un libretto d'istruzioni per usarlo correttamente.
Oltre al file .keras, dobbiamo gestire gli Artifact
- Label Mapping: oltre al modello è fondamentale salvare un file JSON con la mappatura degli indici alle classi testuali per interpretare correttamente l'output della Softmax. La rete ci resstituisce un numero, ci dice 42, ma l'utent vuole leggere 'gatto', quel dizionario di mappatura deve viaggiare insieme al modello, altrimenti Flask non sa come ricostruirli
- Custom Object: se il modello utilizza layer o funzioni di attivazione personalizzate, queste devono essere registrate durante il caricamente tramite 'custom_objects'
- Model Versioning: in produzione è buona norma includere nel nome del file il numero di versione o la data per evitare conflitti durante gli aggiornamenti.

C'è però un dettaglio terncio che spesso viene dimenticato: la normalizzazione

Normalizzazione degli Input in Produzione
Garantire la consistenza numerica
Un errore comuna nel deployment è dimenticare che l'immagine ricevuta via API deve subire lo stesso identico pre-processing usato durante il training, inclusa la normalizzazione dei pixel.
Per ResNet, solitamente si scala l'input nell'intervallo richiesta dal modello, sottraendo la media dei canali RGB dal dataset originale.
In produzione l'immagine che arriva è una griglia di numeri da 0 a 255, ma ResNet è stata addestrata su dati scalati tra -1 e 1, se dimenti questo passaggio il modello vede colori e contrasti che non riconosce, restituendo risultati errati.
Questa equazione deve essere scolpita nel nostro codice di preprocessing, per garantire che l'input sia sempre consistente con ciò che il modello si aspetta.

Ora che il modello è salvo, dobbiamo dargli una porta di accesso, qui entra in gioco Flask

API REST per Computer Vision
Creazione di un endpoint Flask per l'invio di immagini
Flask è un micro framework, ci permette di esporre una funzione di predizione tramite il protocollo HTTP. Invece di parametri testuali, il nostro endpoint dovrà essere in grado di gestire file binari inviati tramite richieste POST.
Flask agisce come un cameriere, riceve gli ordini (le richieste HTTP), li porta in cucine (il nostro modello ResNet) e vi riporta il piatto pronto (la predizione).
In un contesto di computer vision non invieremo testo, ma file binari. useremo il metodo post
Vedremo come ricevere il file, convertirlo in un formato leggibile da Pillow e trasformarlo nel tensore quadridimensionale atteso da ResNet.

Vediamo come si struttura, tecnicamente, questo cameriere digitale

Anatomia del Server di Visione
Flusso della richiesta dal client al modello.
L'anatomia del server è lineare: usiamo i decoratori per gli indirizzi, quando un client invia un immagine accediamo ai dati binari tramite REQUEST.files, è qui che facciamo il ponte, leggiamo immagine con pillow, la trasformiamo in un array Numpy aggiungendo la dimensione del bath (perchè ResNet si aspetta un sensore 4D anche se inviamo una sola immagina alla volta).

- @app.route: decoratore per definire l'URL dell'endpoint (es /predict) e i metodi accettati (POST)
- request.files: accesso ai dati binari dell'immagine caricata dal client senza salvarli su disco
- Image Preprocessing: ridimensionamento del frame a 244x244 pixel e conversione in array Numpy
- JSON Response: restituzione della classe predetta e dalle confidenza in un formato leggibile dalle applicazioni front-end

Il codice alla fine è facile, ma scriverlo bene per non fare esplodere il server, è più complicato

Ottimizzazione del Codice Flask
L'ottimizzazione è vitale, mai caricare il modello all'interno della funzione della rotta. Caricare una Resnet richiede tempo e memoria, farlo ad ogni click dell'utente renderebbe il servizio lentissimo. Dobbiamo caricarlo all'avvio del server una sola volta. Inoltre dobbiamo controllare sempre il file prima di elaborarlo.

- Il modello deve essere caricato una sola volta all'avvio del server (Global Scope) e non all'interno della rotta, per evitare latenze enormi ad ogni richiesta (Model Pre-loading)
- Dobbiamo verificare che il file ricevuto sia effettivamente un'immagine (formato .jpg o .png) prima di tentare la decodifica tensoriale (input Validation)
- E' utile implementare una logica che scarti predizioni con confidenza troppo bassa, restituendo un messaggio di incertezza all'utente (confidenza minima)

Ma come interpretiamo quel numero che esce dalla rete?

Probabilità e Decisione
Interpretazione del vettore Softmax
L'output del modello è un vettore di probabilità la cui somma è unitaria (risultato della softmax), è una distribuzione di probabilità. Noi prendiamo l'indice con valore massimo (argmax), quel valore massimo è anche la nostra misura di certezza.
Estrarre il valore numerico della confidenza permette di quantificare l'affidabilità del sistema in tempo reale.

Gestione del Carico e Concorrenza
Oltre il server di sviluppo: scalare l'applicazione
Il server integrado di Flask è single-threaded e pensato solo per il debug. Se due utenti caricano un'immagine simultaneamente, il secondo dovrà attendere che il primo termini l'inferenza della ResNet.
E' come un piccolo sentiero, può passarci una persona alla volta.
Se due utenti inviano una foto contemporanemaente il secondo deve aspetetare che la CPU finisca i calcoli del primo
In produzione questo non è accettabile.
Per scenari reali, utilizziamo server WSGI (Web Server Gateway Interface) come Gunicorn, capaci di gestire pool di processi 'worker' in parallelo.

Ma come gestiamo questo parallelismo?

Parallelismo in Produzione
Saturare l'hardware per minimizzare la latenza.
Gunicorn crea lw worker, ogni worker è un processo indipendente che ha in memoria una copia del vostro modello. Se avete 4 core potete avere più worker che lavorano in parallelo. Attenzione però alla RAM, se create troppi worker il server andrà in crash. Non dimenticare il timeouts, se l'inferenza di blocca, il worker deve essere riavviato per non tenere occupato la corsia inutilmente

- Worker Processes: istanze separate dell'applicazione che girano su core diversi della cPU
- Gunicorn: server HTTP professionale che agisce da supervisore per i processi Flask
- Memory Management: attenzione al consumo di RAM; ogni worker carica una copia completa del modello neurale.
- Timeouts: gestione dei tempi massimi di attesa per evitare che richiesta bloccate saturino il sistema.

Esistono altre regole d'oro

Best Practies di Deployment
Prima di tutto la sicurezza. Usate un reverse Proxy per gestire le connessioni ed il buffering
Poi seguite il principio dello Stateless Design, il vostro server non deve ricordare le immagini passate, questo permette di scalare orizzontalmente, se il traffico raddoppia basta accendere un nuovo server

- Reverse Proxy: si raccomando l'uso di Ngnix davanti a Gunicorn per gestire buffering delle immagini pesanti e la sicurezza SSL
- Inference Batching: in sistemi ad altissimo traffico, si raggruppano le richieste indivisuali per processarle in un unico batch sulla GPU, massimizzando il throughput.
- Stateless Design: l'API non deve memorizzare stato locale; questo permette di scalare orizzontalmente aggiungendo nuovi server dietro un load balancer.

Ma come facciamo a prevedere quanto traffico può reggere il nostro sistema?

Little's Law e Latenza
Modellazione del throughput del server
Il numero medio di richieste nel sistema è legato al tasso di arrivo e al tempo medio di elaborazione dell'inferenza della ResNet.
Se la vostra Resnet è troppo lenta, potete aggiungere più worker o ottimizzare il modello, magari usando la quantizzazione.
Ridurre il tempo di inferenza, di solito è il modo più efficace per aumentare la capacità del server.

In [ ]:
# python -m pip install flask
import os
from flask import Flask, request, jsonify
import numpy as np
from PIL import Image
import io

# --- CONFIGURAZIONE DELL'AMBIENTE ---

# Impostiamo PyTorch come motore (backend) per Keras.
# Questo influisce su come le operazioni matematiche del modello vengono eseguite.
os.environ["KERAS_BACKEND"] = "torch"

# Disabilitiamo l'uso della GPU per questa istanza per garantire compatibilità 
# o se si preferisce eseguire l'inferenza solo su CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import keras

# Inizializziamo l'applicazione Flask che gestirà le richieste HTTP.
app = Flask(__name__)

# --- GESTIONE DEI PERCORSI E CARICAMENTO DEL MODELLO ---

# Determiniamo dinamicamente la cartella in cui si trova questo script.
# Questo permette di trovare il file del modello indipendentemente da dove viene lanciato il programma.
base_path = os.path.dirname(os.path.abspath(__file__))

# Costruiamo il percorso completo verso il file del modello 'resnet50.keras'.
model_path = os.path.join(base_path, "resnet50.keras")

print(f"Caricamento modello da: {model_path}")

# Carichiamo il modello pre-addestrato ResNet50 salvato precedentemente.
# Il modello contiene l'architettura e i pesi necessari per la classificazione.
model = keras.models.load_model(model_path)

# Importiamo le funzioni specifiche di ResNet50 per la preparazione dei dati e l'interpretazione dei risultati.
from keras.applications.resnet50 import preprocess_input, decode_predictions

# --- DEFINIZIONE DEGLI ENDPOINT ---

@app.route('/predict', methods=['POST'])
def predict():
    """
    Questo endpoint riceve un'immagine tramite una richiesta POST,
    la elabora e restituisce le prime 3 predizioni del modello.
    """
    
    # Controlliamo che nella richiesta ci sia effettivamente un file chiamato 'file'.
    if 'file' not in request.files:
        return jsonify({"error": "Nessun file caricato"}), 400
    
    # Leggiamo i dati binari del file caricato.
    file = request.files['file'].read()
    
    # Trasformiamo i dati binari in un oggetto immagine PIL:
    # 1. io.BytesIO(file) crea un buffer in memoria dai dati binari.
    # 2. Image.open apre il buffer come immagine.
    # 3. .convert('RGB') assicura che l'immagine abbia 3 canali colore (Rosso, Verde, Blu).
    # 4. .resize((224, 224)) adatta l'immagine alla dimensione richiesta da ResNet50.
    img = Image.open(io.BytesIO(file)).convert('RGB').resize((224, 224))
    
    # Trasformiamo l'oggetto immagine PIL in un array NumPy (matrice di numeri).
    x = keras.utils.img_to_array(img)
    
    # ResNet50 si aspetta un "batch" di immagini, quindi una matrice a 4 dimensioni.
    # Usiamo expand_dims per aggiungere la dimensione del batch all'inizio (da (224,224,3) a (1,224,224,3)).
    x = np.expand_dims(x, axis=0)
    
    # Applichiamo il pre-processing specifico di ResNet50 (es. normalizzazione dei colori).
    x = preprocess_input(x)

    # Eseguiamo l'inferenza: il modello analizza l'immagine e restituisce le probabilità per ogni classe.
    preds = model.predict(x)
    
    # Convertiamo i risultati numerici in etichette leggibili (es. "Golden Retriever").
    # top=3 indica che vogliamo solo i primi 3 risultati più probabili.
    results = decode_predictions(preds, top=3)[0]
    
    # Restituiamo i risultati in formato JSON per l'applicazione chiamante.
    return jsonify({
        "predictions": [{"label": r[1], "score": float(r[2])} for r in results]
    })

# --- AVVIO DELL'APPLICAZIONE ---

if __name__ == '__main__':
    # Avviamo il server Flask sulla porta 5000.
    # L'app rimarrà in ascolto per nuove richieste finché non viene interrotta.
    app.run(port=5000)

# Esempi per testare l'endpoint con il comando curl da terminale:
# curl -X POST http://127.0.0.1:5000/predict -F "file=@c:/Percorso/Della/Tua/Immagine.jpg"
# curl -X POST http://127.0.0.1:5000/predict -F "file=@Dog_Breeds.jpeg"

NameError: name '__file__' is not defined